# Warm-up – Threads vs. Processes (and why pandas is slow)

Before we compare big-data engines, we need one fact about Python:
**the Global Interpreter Lock (GIL) lets only one thread execute Python code at a time.**

We test three kinds of work:
1. **Waiting** (chores) – nothing to compute, just waiting
2. **CPU-bound** – pure computation
3. **I/O-bound** – downloading web pages

⚠️ This notebook uses `multiprocessing` with functions defined in the notebook. That works in our
Linux dev container, but often fails on local Windows/macOS Jupyter – one more reason we use the container.

In [ ]:
import time, os, threading, multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import matplotlib.pyplot as plt
print("CPU cores:", os.cpu_count())

## 1. Household chores – single thread vs. threads vs. processes

In [ ]:
CHORES = {"walk the dog": 8, "take out the trash": 2, "get the mail": 4}

def chore(args):
    name, seconds, t0 = args
    start = time.perf_counter() - t0
    time.sleep(seconds)                      # waiting, not computing
    return name, start, time.perf_counter() - t0   # measured INSIDE the task

def run(mode):
    t0 = time.perf_counter()
    jobs = [(n, s, t0) for n, s in CHORES.items()]
    if mode == "single thread":
        return [chore(j) for j in jobs]
    Pool = ThreadPoolExecutor if mode == "threads" else ProcessPoolExecutor
    with Pool(len(jobs)) as ex:
        return list(ex.map(chore, jobs))

fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
for ax, mode in zip(axes, ["single thread", "threads", "processes"]):
    res = run(mode)
    ax.barh([r[0] for r in res], [r[2] - r[1] for r in res], left=[r[1] for r in res])
    ax.set_title(f"{mode}: {max(r[2] for r in res):.1f} s total"); ax.grid(axis="x")
axes[-1].set_xlabel("seconds"); plt.tight_layout(); plt.show()

**Question:** Threads and processes are equally fast here. Why doesn't the GIL matter for waiting?

## 2. CPU-bound work – now the GIL hurts

In [ ]:
def cpu_heavy(n):
    total = 0
    for i in range(n):
        total += i
    return total

N, JOBS = 10**7, 4          # increase N if your machine is fast

t = time.perf_counter(); [cpu_heavy(N) for _ in range(JOBS)]
t_serial = time.perf_counter() - t

with ThreadPoolExecutor(JOBS) as ex:
    t = time.perf_counter(); list(ex.map(cpu_heavy, [N] * JOBS))
t_threads = time.perf_counter() - t

with ProcessPoolExecutor(JOBS) as ex:
    t = time.perf_counter(); list(ex.map(cpu_heavy, [N] * JOBS))
t_procs = time.perf_counter() - t

print(f"serial    {t_serial:5.2f} s")
print(f"threads   {t_threads:5.2f} s   (speed-up {t_serial/t_threads:.1f}x)")
print(f"processes {t_procs:5.2f} s   (speed-up {t_serial/t_procs:.1f}x, max possible ≈ {min(JOBS, os.cpu_count())}x)")

**Question:** Why do threads bring no speed-up here? What limits the process speed-up?

## 3. I/O-bound work – threads shine

In [ ]:
import urllib.request
PAGES = ["Big_data", "Apache_Spark", "Apache_Kafka", "Apache_Hadoop", "MapReduce", "Apache_Flink",
         "Data_lake", "Data_warehouse", "Apache_Parquet", "DuckDB", "Polars_(software)", "Kufstein",
         "Innsbruck", "Tyrol_(federal_state)", "Global_interpreter_lock", "Moore's_law"]
URLS = [f"https://en.wikipedia.org/wiki/{p}" for p in PAGES]

def load(url):
    req = urllib.request.Request(url, headers={"User-Agent": "FH-Kufstein-BigData-Course/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            return len(r.read())
    except Exception as e:
        print("failed:", url, e); return 0

t = time.perf_counter(); [load(u) for u in URLS]
print(f"serial        {time.perf_counter()-t:5.2f} s")
for n in [4, 8, 16]:
    with ThreadPoolExecutor(n) as ex:
        t = time.perf_counter(); list(ex.map(load, URLS))
    print(f"{n:2d} threads    {time.perf_counter()-t:5.2f} s")

## Takeaways
| Work type | Use | Why |
|---|---|---|
| waiting / I/O | threads | the GIL is released while waiting |
| CPU-bound Python code | processes | each process has its own GIL – but no shared memory |

**Bridge to the main lab:** pandas runs most operations on one core. Polars (Rust) and DuckDB (C++)
run their work *outside* the GIL on all cores – no multiprocessing tricks needed.
Spark goes one step further: many processes on **many machines**.

*Outlook:* Python 3.13+ offers an experimental free-threaded build without the GIL (PEP 703).